In [3]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')


# 📊 Spam Classifier using XGBoost (GPU)
In this notebook, we build a high-performance spam classifier using:
- GPU-accelerated XGBoost
- Optuna for hyperparameter tuning
- Custom threshold optimization for best F1 score

Let's start by loading and exploring the dataset.


In [4]:
# Cell 2: Load dataset
df = pd.read_csv("/kaggle/input/sms-spam-collection-dataset/spam.csv", encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 🧹 Data Preprocessing
We'll encode labels (`ham`/`spam`) and vectorize the messages using TF-IDF.


In [5]:
# Cell 3: Encode and vectorize
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

X = df['message']
y = df['label']

tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
X_vec = tfidf.fit_transform(X).toarray()

X_train, X_val, y_train, y_val = train_test_split(X_vec, y, test_size=0.2, stratify=y, random_state=42)


## ⚙️ Hyperparameter Tuning using Optuna (with GPU)
We'll optimize for the highest F1 Score on the validation set.


In [6]:
# Cell 4: Optuna objective
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.3),
        'max_depth': trial.suggest_int("max_depth", 3, 10),
        'n_estimators': trial.suggest_int("n_estimators", 100, 1000),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'colsample_bytree': trial.suggest_float("colsample_bytree", 0.5, 1.0),
        'reg_alpha': trial.suggest_float("reg_alpha", 0.0, 10.0),
        'reg_lambda': trial.suggest_float("reg_lambda", 0.0, 10.0),
        'use_label_encoder': False,
        'eval_metric': 'logloss',
        'tree_method': 'gpu_hist'
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_val, y_pred_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    return np.nanmax(f1_scores)


In [7]:
# Cell 5: Run Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best trial:")
print(f"  F1 Score: {study.best_value:.4f}")
print(f"  Params: {study.best_params}")


[I 2025-07-24 11:10:01,717] A new study created in memory with name: no-name-ae081b12-c6e4-4eb3-80e3-cc1dc55377c5
[I 2025-07-24 11:10:03,800] Trial 0 finished with value: 0.8085106333139681 and parameters: {'learning_rate': 0.013004508092747901, 'max_depth': 7, 'n_estimators': 433, 'subsample': 0.5181364146226006, 'colsample_bytree': 0.8169303882562178, 'reg_alpha': 7.027433131778285, 'reg_lambda': 6.357927603827988}. Best is trial 0 with value: 0.8085106333139681.
[I 2025-07-24 11:10:05,670] Trial 1 finished with value: 0.9071428521635204 and parameters: {'learning_rate': 0.2230249003587227, 'max_depth': 9, 'n_estimators': 639, 'subsample': 0.9285062113953482, 'colsample_bytree': 0.6661672893269888, 'reg_alpha': 4.532228724881525, 'reg_lambda': 5.3306588855393935}. Best is trial 1 with value: 0.9071428521635204.
[I 2025-07-24 11:10:07,809] Trial 2 finished with value: 0.902777772783806 and parameters: {'learning_rate': 0.17810359309850093, 'max_depth': 5, 'n_estimators': 846, 'subsamp

Best trial:
  F1 Score: 0.9199
  Params: {'learning_rate': 0.20885600598505683, 'max_depth': 5, 'n_estimators': 948, 'subsample': 0.5964084911003048, 'colsample_bytree': 0.5346730458485512, 'reg_alpha': 2.0753419031630607, 'reg_lambda': 4.363711517273748}


## 🔍 Threshold Tuning
Let's train the best model and find the optimal threshold on validation set.


In [8]:
# Cell 6: Retrain best model + threshold tuning
best_params = study.best_params
best_params.update({'use_label_encoder': False, 'eval_metric': 'logloss', 'tree_method': 'gpu_hist'})

model = XGBClassifier(**best_params)
model.fit(X_train, y_train)
y_val_proba = model.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, y_val_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Best Threshold: {best_threshold:.4f}")
print(f"F1 Score at Best Threshold: {np.max(f1_scores):.4f}")


Best Threshold: 0.2467
F1 Score at Best Threshold: 0.9199


## ✅ Final Model Performance
Using the optimized threshold and hyperparameters, let's evaluate.


In [9]:
# Cell 7: Final predictions
y_pred_final = (y_val_proba >= best_threshold).astype(int)
final_f1 = f1_score(y_val, y_pred_final)
print(f"Final F1 Score: {final_f1:.4f}")


Final F1 Score: 0.9199


# 🧾 Summary

- ✅ TF-IDF used for text vectorization
- ⚡ GPU-powered XGBoost boosted training speed
- 🎯 Optuna found best hyperparameters
- 🔧 Custom threshold gave max F1 = **~0.91+**

This pipeline is now ready for production or deployment.

---
